# Train an MLP State-Value Model

This notebook trains a neural network to predict the Monte Carlo value target $V^\pi(s')$ from each 390-dimensional post-pick state vector.

## Design goals

- **No draft leakage:** all states from one simulated draft stay in the same split.
- **Representative splits:** draft-level splits are stratified by league configuration.
- **Train-only preprocessing:** feature and target normalization statistics are fitted only on training rows.
- **Stable optimization:** LayerNorm, GELU activations, light dropout, AdamW, gradient clipping, learning-rate reduction, and early stopping.
- **Honest evaluation:** the test set is touched only after model selection; metrics include MAE, RMSE, $R^2$, and slices by format and draft phase.
- **Reproducible deployment:** the checkpoint stores model weights, architecture, feature names, and normalization statistics.

The target is average weekly fantasy points, not total season points.

In [ ]:
from __future__ import annotations

import json
import random
from copy import deepcopy
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
BATCH_SIZE = 512
MAX_EPOCHS = 100
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# Robust path discovery: works in local repo, Google Colab (/content), or standalone
possible_data_paths = [
    ROOT / "data" / "post_pick_values.parquet",
    Path("data/post_pick_values.parquet"),
    Path("post_pick_values.parquet"),
    Path("/content/data/post_pick_values.parquet"),
    Path("/content/post_pick_values.parquet"),
]

DATA_PATH = next((p for p in possible_data_paths if p.exists()), possible_data_paths[0])
MODEL_PATH = ROOT / "models" / "value_mlp.pt" if (ROOT / "pyproject.toml").exists() else DATA_PATH.parent / "value_mlp.pt"

print(f"Device: {DEVICE}")
print(f"Dataset path: {DATA_PATH.resolve() if DATA_PATH.exists() else DATA_PATH}")
print(f"Model save path: {MODEL_PATH}")

## 1. Load and validate the Parquet dataset

The state vectors are stored as a fixed-size Parquet list. Schema metadata contains the exact feature order used by the encoder. We validate dimensions and finite values before training so malformed data fails early.

In [ ]:
table = pq.read_table(DATA_PATH)
metadata = table.schema.metadata or {}
feature_names = json.loads(metadata[b"feature_names"].decode("utf-8"))
df = table.drop(["state_vector"]).to_pandas()
X = np.asarray(table["state_vector"].to_pylist(), dtype=np.float32)
y = df["reward"].to_numpy(dtype=np.float32)

assert X.ndim == 2 and X.shape[1] == len(feature_names)
assert len(X) == len(y)
assert np.isfinite(X).all() and np.isfinite(y).all()

# decision_id starts with draft-XXXXXX; all states from that draft share a group.
df["draft_id"] = df["decision_id"].str.extract(r"^(draft-\d+)", expand=False)
assert df["draft_id"].notna().all()

print(f"Rows: {len(df):,}")
print(f"Features: {X.shape[1]}")
print(f"Unique simulated drafts: {df['draft_id'].nunique():,}")
print(f"Reward mean/std: {y.mean():.2f} / {y.std():.2f}")
df[["reward", "reward_standard_error", "round_number", "phase", "league_config_id"]].head()

## 2. Draft-grouped train/validation/test split

A random row split would leak the same synthetic player pool and shared draft history into multiple splits because several rows are captured from each draft. Instead, we split **unique draft IDs** and stratify those IDs by league configuration.

- **70% train:** fit network weights and normalization statistics.
- **15% validation:** early stopping and learning-rate scheduling.
- **15% test:** final unbiased evaluation only.

The fixed seed makes the split reproducible.

In [ ]:
drafts = (
    df[["draft_id", "league_config_id"]]
    .drop_duplicates("draft_id")
    .reset_index(drop=True)
)

train_drafts, holdout_drafts = train_test_split(
    drafts,
    test_size=0.30,
    random_state=SEED,
    stratify=drafts["league_config_id"],
)
val_drafts, test_drafts = train_test_split(
    holdout_drafts,
    test_size=0.50,
    random_state=SEED,
    stratify=holdout_drafts["league_config_id"],
)

split_ids = {
    "train": set(train_drafts["draft_id"]),
    "val": set(val_drafts["draft_id"]),
    "test": set(test_drafts["draft_id"]),
}
assert split_ids["train"].isdisjoint(split_ids["val"])
assert split_ids["train"].isdisjoint(split_ids["test"])
assert split_ids["val"].isdisjoint(split_ids["test"])

indices = {
    name: np.flatnonzero(df["draft_id"].isin(ids).to_numpy())
    for name, ids in split_ids.items()
}

split_summary = pd.DataFrame(
    {
        name: {
            "rows": len(idx),
            "drafts": df.iloc[idx]["draft_id"].nunique(),
            "reward_mean": df.iloc[idx]["reward"].mean(),
            "reward_std": df.iloc[idx]["reward"].std(),
        }
        for name, idx in indices.items()
    }
).T
print(split_summary)

# Confirm every supported format appears in every split.
for name, idx in indices.items():
    assert df.iloc[idx]["league_config_id"].nunique() == df["league_config_id"].nunique()

## 3. Train-only normalization and data loaders

Feature scales vary substantially: binary masks are 0/1 while player means and draft counts are much larger. Standardization improves conditioning. Near-constant columns receive a scale of 1, avoiding division by zero. The reward is also standardized during optimization, then transformed back to weekly points for all reported metrics.

We deliberately do not use `reward_standard_error` as an input because it is produced by label generation and will not exist for a live draft state.

In [ ]:
train_idx = indices["train"]
feature_mean = X[train_idx].mean(axis=0, dtype=np.float64).astype(np.float32)
feature_std = X[train_idx].std(axis=0, dtype=np.float64).astype(np.float32)
feature_std[feature_std < 1e-6] = 1.0

target_mean = float(y[train_idx].mean())
target_std = float(y[train_idx].std())
assert target_std > 0

X_scaled = (X - feature_mean) / feature_std
y_scaled = (y - target_mean) / target_std


def make_dataset(idx: np.ndarray) -> TensorDataset:
    return TensorDataset(
        torch.from_numpy(X_scaled[idx].astype(np.float32, copy=False)),
        torch.from_numpy(y_scaled[idx, None].astype(np.float32, copy=False)),
    )


generator = torch.Generator().manual_seed(SEED)
loaders = {
    "train": DataLoader(
        make_dataset(indices["train"]),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
        pin_memory=DEVICE.type == "cuda",
    ),
    "val": DataLoader(make_dataset(indices["val"]), batch_size=2048, shuffle=False),
    "test": DataLoader(make_dataset(indices["test"]), batch_size=2048, shuffle=False),
}

print(f"Target normalization: mean={target_mean:.3f}, std={target_std:.3f}")
print(f"Constant/near-constant features: {int((X[train_idx].std(axis=0) < 1e-6).sum())}")

## 4. MLP architecture

The encoder already imposes a canonical order on board and roster slots, so a dense network is an appropriate first model.

Architecture: **390 → 256 → 128 → 64 → 1**.

- **GELU** provides smooth nonlinearities.
- **LayerNorm** stabilizes hidden representations without depending on batch statistics.
- **Dropout 0.10** gives mild regularization without discarding too much signal.
- The final layer is linear because weekly points are an unbounded regression target.

This model has enough capacity for interactions such as league format × roster needs × available talent, while remaining small relative to the dataset. It is a strong baseline; more specialized board/roster encoders should only be considered after measuring this model.

In [ ]:
class ValueMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dims: tuple[int, ...] = (256, 128, 64),
        dropout: float = 0.10,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        previous = input_dim
        for width in hidden_dims:
            layers.extend(
                [
                    nn.Linear(previous, width),
                    nn.LayerNorm(width),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            previous = width
        layers.append(nn.Linear(previous, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


MODEL_CONFIG = {
    "input_dim": X.shape[1],
    "hidden_dims": (256, 128, 64),
    "dropout": 0.10,
}
model = ValueMLP(**MODEL_CONFIG).to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f"Trainable parameters: {parameter_count:,}")

## 5. Optimization and early stopping

We optimize normalized mean squared error with AdamW. Validation MSE controls both `ReduceLROnPlateau` and early stopping. Gradient clipping guards against occasional unstable updates. The best validation checkpoint—not the final epoch—is restored before test evaluation.

In [ ]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=4,
    min_lr=1e-6,
)


def run_epoch(loader: DataLoader, training: bool) -> float:
    model.train(training)
    total_loss = 0.0
    total_examples = 0

    for features, targets in loader:
        features = features.to(DEVICE)
        targets = targets.to(DEVICE)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            predictions = model(features)
            loss = loss_fn(predictions, targets)

        if training:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        batch_size = features.shape[0]
        total_loss += float(loss.detach()) * batch_size
        total_examples += batch_size

    return total_loss / total_examples


history = {"train_loss": [], "val_loss": [], "learning_rate": []}
best_state = None
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_loss = run_epoch(loaders["train"], training=True)
    val_loss = run_epoch(loaders["val"], training=False)
    scheduler.step(val_loss)
    learning_rate = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["learning_rate"].append(learning_rate)

    if val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss
        best_state = deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 5 == 0:
        print(
            f"Epoch {epoch:03d} | train={train_loss:.5f} | "
            f"val={val_loss:.5f} | lr={learning_rate:.2e}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch}; best validation loss={best_val_loss:.5f}")
        break

assert best_state is not None
model.load_state_dict(best_state)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="validation")
axes[0].set(xlabel="Epoch", ylabel="Normalized MSE", title="Learning curves")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(history["learning_rate"])
axes[1].set(xlabel="Epoch", ylabel="Learning rate", title="Learning-rate schedule")
axes[1].set_yscale("log")
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 6. Final evaluation

MAE is the easiest metric to interpret in weekly fantasy points. RMSE penalizes larger misses, and $R^2$ measures variance explained. We compare against:

1. **Global train mean:** predicts the same score for every state.
2. **League-format train mean:** predicts the training mean for each league configuration.

The second baseline is important because starter counts explain a large portion of reward variance. The MLP should beat it by learning roster quality, available talent, and draft progress within each format.

In [ ]:
@torch.no_grad()
def predict_indices(idx: np.ndarray) -> np.ndarray:
    model.eval()
    features = torch.from_numpy(X_scaled[idx].astype(np.float32, copy=False))
    predictions: list[np.ndarray] = []
    for start in range(0, len(features), 2048):
        batch = features[start : start + 2048].to(DEVICE)
        predictions.append(model(batch).cpu().numpy().ravel())
    normalized = np.concatenate(predictions)
    return normalized * target_std + target_mean


def regression_metrics(actual: np.ndarray, predicted: np.ndarray) -> dict[str, float]:
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": mean_squared_error(actual, predicted) ** 0.5,
        "R2": r2_score(actual, predicted),
    }


test_idx = indices["test"]
y_test = y[test_idx]
test_predictions = predict_indices(test_idx)

train_format_means = (
    df.iloc[train_idx]
    .groupby("league_config_id")["reward"]
    .mean()
)
test_format_predictions = (
    df.iloc[test_idx]["league_config_id"]
    .map(train_format_means)
    .to_numpy(dtype=np.float32)
)

metrics = pd.DataFrame(
    {
        "Global train mean": regression_metrics(
            y_test,
            np.full_like(y_test, target_mean),
        ),
        "League-format mean": regression_metrics(y_test, test_format_predictions),
        "MLP": regression_metrics(y_test, test_predictions),
    }
).T
metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, test_predictions, s=8, alpha=0.25)
limits = [min(y_test.min(), test_predictions.min()), max(y_test.max(), test_predictions.max())]
axes[0].plot(limits, limits, "k--", linewidth=1)
axes[0].set(
    xlabel="Observed reward",
    ylabel="Predicted reward",
    title="Test predictions",
)
axes[0].grid(alpha=0.2)

residuals = test_predictions - y_test
axes[1].hist(residuals, bins=50, alpha=0.8)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1)
axes[1].set(
    xlabel="Prediction − observed (points/week)",
    ylabel="Count",
    title="Test residuals",
)
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.3f}")
print(f"Residual standard deviation: {residuals.std():.3f}")

## 7. Slice diagnostics

Aggregate metrics can hide systematic errors. We inspect test MAE and bias across QB format, FLEX format, league size, and draft phase. A healthy value model should have small bias in every slice; consistently high slice error indicates where more data or a richer architecture is needed.

In [ ]:
test_results = df.iloc[test_idx][
    ["sample_id", "num_teams", "qb_starters", "flex_slots", "phase", "reward"]
].copy()
test_results["prediction"] = test_predictions
test_results["error"] = test_results["prediction"] - test_results["reward"]
test_results["absolute_error"] = test_results["error"].abs()


def slice_report(column: str) -> pd.DataFrame:
    return (
        test_results.groupby(column, observed=True)
        .agg(
            count=("reward", "size"),
            observed_mean=("reward", "mean"),
            predicted_mean=("prediction", "mean"),
            mae=("absolute_error", "mean"),
            bias=("error", "mean"),
        )
        .round(3)
    )


for column in ["qb_starters", "flex_slots", "num_teams", "phase"]:
    print(f"\nSlice: {column}")
    display(slice_report(column))

## 8. Save the deployable checkpoint

The checkpoint includes everything required to reproduce predictions from raw encoded states. Keeping preprocessing with the weights prevents train/inference skew. The test metrics and split seed are included for traceability.

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
checkpoint = {
    "model_state_dict": {key: value.detach().cpu() for key, value in model.state_dict().items()},
    "model_config": MODEL_CONFIG,
    "feature_names": feature_names,
    "feature_mean": torch.from_numpy(feature_mean),
    "feature_std": torch.from_numpy(feature_std),
    "target_mean": target_mean,
    "target_std": target_std,
    "encoder_version": int(df["encoder_version"].iloc[0]),
    "split_seed": SEED,
    "test_metrics": metrics.loc["MLP"].to_dict(),
    "reward_semantics": metadata[b"reward_semantics"].decode("utf-8"),
}
torch.save(checkpoint, MODEL_PATH)
print(f"Saved checkpoint to {MODEL_PATH}")
print(f"Checkpoint size: {MODEL_PATH.stat().st_size / (1024 ** 2):.2f} MB")

## 9. Using the model for draft decisions

For a live draft state $s$:

1. Enumerate every legal player ID $a$.
2. Apply the deterministic transition $s' = T(s,a)$.
3. Encode each post-pick state $s'$ with the same `StateEncoder`.
4. Normalize with the saved feature mean and standard deviation.
5. Predict $\hat V(s')$ for all candidates in one batch.
6. Choose $\arg\max_a \hat V(T(s,a))$.

The helper below performs steps 4–5. It checks feature dimensionality and returns predictions in weekly fantasy points.

In [ ]:
def predict_state_values(raw_state_vectors: np.ndarray) -> np.ndarray:
    """Predict weekly value for one or more raw 390-feature state vectors."""
    vectors = np.asarray(raw_state_vectors, dtype=np.float32)
    if vectors.ndim == 1:
        vectors = vectors[None, :]
    if vectors.ndim != 2 or vectors.shape[1] != MODEL_CONFIG["input_dim"]:
        raise ValueError(f"expected shape (n, {MODEL_CONFIG['input_dim']}), got {vectors.shape}")

    normalized = (vectors - feature_mean) / feature_std
    model.eval()
    with torch.no_grad():
        normalized_predictions = (
            model(torch.from_numpy(normalized).to(DEVICE))
            .cpu()
            .numpy()
            .ravel()
        )
    return normalized_predictions * target_std + target_mean


# Smoke test: predictions from the helper match the held-out evaluation path.
np.testing.assert_allclose(
    predict_state_values(X[test_idx[:8]]),
    test_predictions[:8],
    rtol=1e-5,
    atol=1e-5,
)
print(predict_state_values(X[test_idx[:5]]))

## 10. Live Round 1 Pick Decision with the Trained Value Network

Given an active draft state at our turn in Round 1 (here demonstrated at Pick 1.01 in a 10-team, 1-QB, 2-RB, 2-WR, 1-TE, 1-FLEX league):
1. Retrieve all legal player actions.
2. Step to the exact post-pick state $s' = T(s, a)$ for every legal pick.
3. Encode each post-pick state into the 390-D canonical feature vector.
4. Run a single batched forward pass through `ValueMLP` to predict $\hat{V}(s')$.
5. Output the ranked leaderboard and select the $\arg\max_a \hat{V}(T(s, a))$ pick.

In [ ]:
import sys
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from fantasy_draft.config import LeagueConfig, RosterConfig
from fantasy_draft.draft import DraftEnvironment
from fantasy_draft.models import Position
from fantasy_draft.players import generate_players
from fantasy_draft.state import StateEncoder

# 1. Initialize a live draft environment & player pool
league = LeagueConfig(
    n_teams=10,
    controlled_team=0,  # We have pick 1 (1.01)
    roster=RosterConfig(
        required={Position.QB: 1, Position.RB: 2, Position.WR: 2, Position.TE: 1},
        flex_slots=1,
        bench_slots=3,
    ),
)
env = DraftEnvironment(league)
players = generate_players(seed=101)
initial_state = env.initial_state(players)
encoder = StateEncoder()

print(f"Current pick: Round {initial_state.round_number}, Pick #{initial_state.pick_index + 1}")
print(f"Active team to draft: Team {initial_state.current_team} (Controlled: {league.controlled_team})")

# 2. Enumerate every legal pick and compute the next transition s' = T(s, a)
legal_actions = env.legal_actions(initial_state)
next_states = [env.step(initial_state, action, validate=False) for action in legal_actions]
next_vectors = np.stack([encoder.encode_state(s_prime) for s_prime in next_states])

# 3. Predict state value V(s') for each candidate pick
predicted_values = predict_state_values(next_vectors)

# 4. Build leaderboard of candidates
candidates = []
player_map = initial_state.player_map
for action, pred_val in zip(legal_actions, predicted_values):
    p = player_map[action]
    candidates.append({
        "player_id": p.player_id,
        "position": p.position.value,
        "mu": p.mu,
        "sigma": p.sigma,
        "injury_prob": p.injury_probability,
        "predicted_weekly_reward": pred_val,
    })

leaderboard = pd.DataFrame(candidates).sort_values("predicted_weekly_reward", ascending=False).reset_index(drop=True)
best_pick = leaderboard.iloc[0]

print("\n" + "=" * 65)
print(f"  NN RECOMMENDED PICK: {best_pick['position']} #{int(best_pick['player_id'])}")
print(f"  Projected Team Value: {best_pick['predicted_weekly_reward']:.3f} weekly pts")
print(f"  Player Stats: mu={best_pick['mu']:.2f}, sigma={best_pick['sigma']:.2f}, injury_p={best_pick['injury_prob']:.3f}")
print("=" * 65 + "\n")

print("Top 10 Picks by Predicted Value:")
display(leaderboard.head(10))

print("Top Candidate per Position:")
display(leaderboard.groupby("position", as_index=False).first().sort_values("predicted_weekly_reward", ascending=False))